# Epic 4: Model Building

In [ ]:
import osimport pickleimport pandas as pdfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.neighbors import KNeighborsClassifierfrom xgboost import XGBClassifierfrom sklearn.metrics import accuracy_score, classification_report, confusion_matriximport matplotlib.pyplot as pltDATA_PATH = os.path.join("..", "data", "flood dataset.xlsx")df = pd.read_excel(DATA_PATH)FEATURES = ["Temp", "Humidity", "Cloud Cover", "ANNUAL", "Jun-Sep"]X = df[FEATURES]y = df["flood"]X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)scaler = StandardScaler()X_train_s = scaler.fit_transform(X_train)X_test_s = scaler.transform(X_test)

## Story 1: Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42, max_depth=5)dt.fit(X_train_s, y_train)dt_acc = accuracy_score(y_test, dt.predict(X_test_s))print(f"Decision Tree Accuracy: {dt_acc:.4f}")print(classification_report(y_test, dt.predict(X_test_s)))

## Story 2: Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)rf.fit(X_train_s, y_train)rf_acc = accuracy_score(y_test, rf.predict(X_test_s))print(f"Random Forest Accuracy: {rf_acc:.4f}")

## Story 3: KNN

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)knn.fit(X_train_s, y_train)knn_acc = accuracy_score(y_test, knn.predict(X_test_s))print(f"KNN Accuracy: {knn_acc:.4f}")

## Story 4: XGBoost

In [ ]:
xgb = XGBClassifier(random_state=42, eval_metric="logloss", n_estimators=150, max_depth=4, scale_pos_weight=6)xgb.fit(X_train_s, y_train)xgb_acc = accuracy_score(y_test, xgb.predict(X_test_s))print(f"XGBoost Accuracy: {xgb_acc:.4f}")print(classification_report(y_test, xgb.predict(X_test_s)))

## Story 5: Compare all models

In [ ]:
results = {"Decision Tree": dt_acc, "Random Forest": rf_acc, "KNN": knn_acc, "XGBoost": xgb_acc}for name, acc in sorted(results.items(), key=lambda x: -x[1]):    print(f"{name}: {acc*100:.2f}%")plt.bar(results.keys(), [v*100 for v in results.values()], color=["#0d6efd", "#198754", "#ffc107", "#dc3545"])plt.ylabel("Accuracy (%)")plt.title("Model Comparison")plt.ylim(80, 100)plt.savefig("../outputs/model_comparison.png", dpi=120)plt.show()

## Story 6: Save best model

In [ ]:
best_name = max(results, key=results.get)if results.get("XGBoost", 0) >= results[best_name] - 0.02:    best_name = "XGBoost"models = {"Decision Tree": dt, "Random Forest": rf, "KNN": knn, "XGBoost": xgb}best_model = models[best_name]artifact = {"model": best_model, "scaler": scaler, "features": FEATURES, "model_name": best_name, "accuracy": results[best_name], "all_results": results}model_path = os.path.join("..", "app", "models", "best_model.pkl")os.makedirs(os.path.dirname(model_path), exist_ok=True)with open(model_path, "wb") as f:    pickle.dump(artifact, f)print(f"Saved {best_name} to {model_path}")